# Survey of the Hindi Wikipedia dump

Reads the compressed dump as a stream (nothing is decompressed to disk) and reports:

* the wiki, schema version, and range of page IDs in the file
* pages per namespace, with the namespace names from `<siteinfo>`
* in namespace 0: redirects (checked against #REDIRECT, #अनुप्रेषित and #पुनर्प्रेषित), pages with empty text, content models, and revisions per page
* length of the remaining articles (characters of wikitext)
* groups of articles with the same skeleton (templates, title, links and numbers masked), as a lower bound on bot-made stubs
* the most frequent section headings at every level, after removing bold/italic, HTML tags, a trailing colon or danda, and invisible format characters
* the most frequent templates, with redirects to a template counted under the template itself
* optionally, how many of those articles are flagged as disambiguation pages in the matching `page_props` SQL dump

HTML comments and the contents of `<nowiki>`, `<pre>`, `<syntaxhighlight>`, `<source>` and `<math>` are ignored when looking for headings and templates.

Works with both the legacy pages-articles dump and the new MediaWiki Content File Export; the schema version is read from the file.

In [18]:
import bz2
import gzip
import hashlib
import json
import re
import statistics
import sys
import time
import unicodedata
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

## Patterns

In [19]:
# A heading is a whole line "= Title =" to "====== Title ======"; group 1 gives the level.
# [^\S\n] is whitespace other than a newline, so a heading never spans two lines.
HEADING_RE = re.compile(r"^(={1,6})[^\S\n]*(.+?)[^\S\n]*\1[^\S\n]*$", re.M)
# Template name: text after "{{" up to the first "|" or "}}" (skips "{{{param}}}").
TEMPLATE_RE = re.compile(r"(?<!\{)\{\{(?!\{)\s*([^|{}\n]+?)\s*(?:\||\}\})")
# A template with no template inside it; removed repeatedly to strip nested templates.
INNERMOST_TEMPLATE_RE = re.compile(r"\{\{[^{}]*\}\}")
# HTML comments (an unclosed one runs to the end, as in MediaWiki) and tags whose contents
# are not wikitext. Self-closing tags such as <nowiki /> are left alone.
UNPARSED_RE = re.compile(
    r"<!--.*?(?:-->|\Z)|<(nowiki|pre|syntaxhighlight|source|math)\b[^>/]*>.*?</\1\s*>",
    re.S | re.I,
)
# Bold/italic quotes and HTML tags inside a heading, e.g. "== '''सन्दर्भ''' ==".
HEADING_MARKUP_RE = re.compile(r"'{2,}|<[^>]*>")
# Redirect magic words: the English one and the two Hindi aliases.
REDIRECT_TEXT_RE = re.compile(r"\s*#\s*(?:REDIRECT|अनुप्रेषित|पुनर्प्रेषित)", re.I)
# Links and numbers (including Devanagari digits), masked in an article's skeleton.
SKELETON_RE = re.compile(r"\[\[[^\[\]]*\]\]|\d+")
# One (page_id, 'property_name', ...) row in a page_props INSERT statement.
PAGE_PROPS_ROW_RE = re.compile(r"\((\d+),'((?:[^'\\]|\\.)*)',")

## Helpers

In [20]:
def normalise_label(s: str) -> str:
    """NFC, format characters (ZWSP, ZWNJ, ZWJ, soft hyphen, LRM/RLM, BOM) removed,
    whitespace collapsed. str.split() treats U+00A0 and the other Unicode spaces as
    whitespace, so they become ordinary spaces."""
    s = unicodedata.normalize("NFC", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Cf")
    return " ".join(s.split())


def clean_heading(raw: str) -> str:
    """Heading text without bold/italic quotes, HTML tags, or a trailing colon or danda;
    empty if nothing but '=' is left."""
    s = normalise_label(HEADING_MARKUP_RE.sub("", raw)).rstrip(" :।")
    return s if s.strip("=") else ""


def normalise_template(name: str) -> str:
    """Template names ignore first-letter case and treat '_' as a space."""
    name = normalise_label(name.replace("_", " "))
    return name[:1].upper() + name[1:]


def strip_template_prefix(name: str, prefixes: set) -> str:
    """'साँचा:Foo' or 'Template:foo' -> 'Foo'; other names are returned unchanged."""
    prefix, colon, rest = name.partition(":")
    if colon and normalise_template(prefix) in prefixes and rest.strip():
        return normalise_template(rest)
    return name

In [21]:
def skeleton_key(title: str, text: str) -> bytes:
    """Hash of the wikitext with templates removed and the title, links and numbers
    masked, so bot-made stubs that differ only in those get the same key."""
    while True:
        stripped = INNERMOST_TEMPLATE_RE.sub("", text)
        if stripped == text:
            break
        text = stripped
    if title:
        text = text.replace(title, "@")
    text = " ".join(SKELETON_RE.sub("#", text).split())
    return hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest()


def merge_template_redirects(templates: Counter, redirect_titles: list, prefixes: set):
    """Count each template under the name its redirects point to.

    redirect_titles holds (redirect page title, target title) pairs from namespace 10.
    Returns (counts, aliases). A merged count is an upper bound, because an article
    that uses two names of the same template is counted once for each."""
    redirects = {
        strip_template_prefix(normalise_template(src), prefixes):
            strip_template_prefix(normalise_template(tgt), prefixes)
        for src, tgt in redirect_titles
    }

    def resolve(name):
        seen = set()
        while name in redirects and name not in seen:      # follow double redirects
            seen.add(name)
            name = redirects[name]
        return name

    counts = Counter()
    aliases = defaultdict(set)
    for name, n in templates.items():
        canonical = resolve(strip_template_prefix(name, prefixes))
        counts[canonical] += n
        if name != canonical:
            aliases[canonical].add(name)
    return counts, aliases

In [22]:
def read_disambiguation_ids(path: str) -> set:
    ids = set()
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            for page_id, prop in PAGE_PROPS_ROW_RE.findall(line):
                if prop == "disambiguation":
                    ids.add(int(page_id))
    return ids

## Survey

In [23]:
def survey(dump_path: str, disambiguation_ids: set, top: int, dup_min: int) -> dict:
    dbname = None
    namespace_names = {}
    pages_per_ns = Counter()
    pages_total = 0
    min_id, max_id = sys.maxsize, 0
    ns0_redirects = 0
    redirect_first_words = Counter()             # redirects without a known magic word
    text_redirects = 0                           # redirect wikitext but no <redirect>
    empty_text = 0
    ns0_models = Counter()
    revisions_per_page = Counter()
    article_lengths = []
    disambiguation_hits = 0
    skeletons = Counter()
    skeleton_example = {}                        # skeleton -> first title with it
    headings = Counter()
    heading_levels = defaultdict(Counter)        # heading -> articles per level
    heading_spellings = defaultdict(Counter)     # heading -> raw forms
    templates = Counter()
    template_redirect_titles = []                # (redirect, target) in namespace 10

    started = time.time()
    with bz2.open(dump_path, "rb") as fh:
        context = ET.iterparse(fh, events=("start", "end"))
        _, root = next(context)                   # the <mediawiki> element
        if not root.tag.endswith("mediawiki"):
            raise ValueError(f"not a MediaWiki export: root element is {root.tag}")
        # Take the schema namespace from the file instead of assuming a version.
        xmlns = root.tag[:root.tag.index("}") + 1] if root.tag.startswith("{") else ""
        for event, elem in context:
            if event != "end":
                continue

            if elem.tag == xmlns + "siteinfo":
                dbname = elem.findtext(xmlns + "dbname")
                for ns in elem.iter(xmlns + "namespace"):
                    namespace_names[ns.get("key")] = ns.text or "(articles)"
                root.clear()
                continue

            if elem.tag != xmlns + "page":
                continue

            ns = elem.findtext(xmlns + "ns")
            title = elem.findtext(xmlns + "title") or ""
            page_id = int(elem.findtext(xmlns + "id"))
            redirect = elem.find(xmlns + "redirect")
            pages_per_ns[ns] += 1
            pages_total += 1
            min_id, max_id = min(min_id, page_id), max(max_id, page_id)

            if ns == "10" and redirect is not None:
                template_redirect_titles.append((title, redirect.get("title", "")))

            if ns == "0":
                revisions = elem.findall(xmlns + "revision")
                revisions_per_page[len(revisions)] += 1
                latest = revisions[-1] if revisions else None
                ns0_models[latest.findtext(xmlns + "model") if latest is not None else None] += 1
                text = (latest.findtext(xmlns + "text") if latest is not None else None) or ""
                looks_like_redirect = REDIRECT_TEXT_RE.match(text) is not None

                if redirect is not None:
                    ns0_redirects += 1
                    if not looks_like_redirect:
                        redirect_first_words[(text.split(None, 1) or ["(empty)"])[0][:40]] += 1
                elif not text.strip():
                    empty_text += 1
                else:
                    if looks_like_redirect:
                        text_redirects += 1
                    article_lengths.append(len(text))
                    if page_id in disambiguation_ids:
                        disambiguation_hits += 1
                    else:
                        skeleton = skeleton_key(title, text)
                        skeletons[skeleton] += 1
                        skeleton_example.setdefault(skeleton, title)
                    scan = UNPARSED_RE.sub("", text)
                    # Count each heading/template once per article.
                    found = {(clean_heading(raw), len(eq), raw) for eq, raw in HEADING_RE.findall(scan)}
                    found = {f for f in found if f[0]}
                    for key in {k for k, _, _ in found}:
                        headings[key] += 1
                    for key, level in {(k, lv) for k, lv, _ in found}:
                        heading_levels[key][level] += 1
                    for key, raw in {(k, r) for k, _, r in found}:
                        heading_spellings[key][raw] += 1
                    for name in set(normalise_template(t) for t in TEMPLATE_RE.findall(scan)):
                        templates[name] += 1

            root.clear()                          # drop the finished page from memory
            if pages_total % 50000 == 0:
                print(f"  ... {pages_total:,} pages read ({time.time() - started:.0f}s)", file=sys.stderr)

    if pages_total == 0:
        raise ValueError(f"no <page> elements found (schema namespace {xmlns!r})")

    prefixes = {normalise_template(namespace_names.get("10", "Template")), "Template"}
    template_counts, template_aliases = merge_template_redirects(templates, template_redirect_titles, prefixes)

    groups = sorted(((n, k) for k, n in skeletons.items() if n >= dup_min), reverse=True)
    in_groups = sum(n for n, _ in groups)

    def raw_forms(h):
        spellings = heading_spellings[h]
        if len(spellings) < 2:
            return []
        return [f"{r!r}{'' if unicodedata.is_normalized('NFC', r) else ' [not NFC]'} ({n:,})"
                for r, n in spellings.most_common(10)]

    lengths = sorted(article_lengths)
    return {
        "dump": dump_path,
        "dbname": dbname,
        "schema": xmlns.strip("{}"),
        "seconds": round(time.time() - started),
        "pages_total": pages_total,
        "page_id_min": min_id,
        "page_id_max": max_id,
        "pages_per_namespace": {
            f"{k} {namespace_names.get(k, '?')}": v for k, v in pages_per_ns.most_common()
        },
        "ns0": {
            "pages": pages_per_ns["0"],
            "redirects": ns0_redirects,
            "redirects_without_magic_word": sum(redirect_first_words.values()),
            "redirect_first_words": dict(redirect_first_words.most_common(10)),
            "non_redirect": len(article_lengths) + empty_text,
            "empty_text": empty_text,
            "articles": len(article_lengths),
            "text_redirects_without_element": text_redirects,
            "flagged_disambiguation": disambiguation_hits if disambiguation_ids else None,
            "content_models": dict(ns0_models),
            "revisions_per_page": dict(revisions_per_page),
        },
        "article_length_chars": {
            "median": statistics.median(lengths) if lengths else None,
            "p10": lengths[len(lengths) // 10] if lengths else None,
            "p90": lengths[9 * len(lengths) // 10] if lengths else None,
            "share_under_500": round(sum(l < 500 for l in lengths) / len(lengths), 3) if lengths else None,
        },
        "same_skeleton": {
            "min_group_size": dup_min,
            "groups": len(groups),
            "articles": in_groups,
            "share": round(in_groups / sum(skeletons.values()), 3) if skeletons else 0,
            "largest": [{"articles": n, "example": skeleton_example[k]} for n, k in groups[:20]],
        },
        "top_headings": [
            {"heading": h, "articles": c,
             "levels": dict(sorted(heading_levels[h].items())),
             "spellings": len(heading_spellings[h]),
             "raw_forms": raw_forms(h)}
            for h, c in headings.most_common(top)
        ],
        "template_redirects": len(template_redirect_titles),
        "top_templates": [
            {"template": t, "articles": c, "aliases": sorted(template_aliases[t])}
            for t, c in template_counts.most_common(top)
        ],
    }

In [24]:
def print_report(r: dict) -> None:
    print(f"\nDump: {r['dump']}  ({r['seconds']} s)")
    print(f"Wiki {r['dbname']}, schema {r['schema']}")
    print(f"Pages in total: {r['pages_total']:,}  (page IDs {r['page_id_min']:,} to {r['page_id_max']:,})\n")
    print("Pages per namespace:")
    for k, v in r["pages_per_namespace"].items():
        print(f"  {v:>9,}  {k}")
    n = r["ns0"]
    print("\nNamespace 0:")
    print(f"  pages               {n['pages']:,}")
    print(f"  redirects           {n['redirects']:,}")
    if n["redirects_without_magic_word"]:
        print(f"    without #REDIRECT/#अनुप्रेषित/#पुनर्प्रेषित: {n['redirects_without_magic_word']:,}, "
              f"first words {n['redirect_first_words']}")
    print(f"  non-redirect        {n['non_redirect']:,}")
    print(f"  empty text          {n['empty_text']:,}")
    print(f"  articles            {n['articles']:,}")
    if n["text_redirects_without_element"]:
        print(f"    redirect wikitext but no <redirect> element: {n['text_redirects_without_element']:,}")
    if n["flagged_disambiguation"] is not None:
        print(f"  disambiguation      {n['flagged_disambiguation']:,}  (from page_props)")
        print(f"  => articles to keep {n['articles'] - n['flagged_disambiguation']:,}")
    print(f"  content models      {n['content_models']}")
    print(f"  revisions per page  {n['revisions_per_page']}")
    a = r["article_length_chars"]
    print(f"\nArticle wikitext length: median {a['median']}, 10th pct {a['p10']}, "
          f"90th pct {a['p90']}, share under 500 chars {a['share_under_500']}")
    s = r["same_skeleton"]
    print(f"\nArticles sharing a skeleton with {s['min_group_size'] - 1}+ others: {s['articles']:,} "
          f"({s['share']:.1%}) in {s['groups']:,} groups  (a lower bound on bot-made stubs)")
    for g in s["largest"]:
        print(f"  {g['articles']:>8,}  e.g. {g['example']}")
    print("\nMost frequent headings, all levels (articles containing each):")
    for h in r["top_headings"]:
        levels = ", ".join(f"L{lv} {c:,}" for lv, c in h["levels"].items())
        extra = f"   [{h['spellings']} spellings: {', '.join(h['raw_forms'])}]" if h["raw_forms"] else ""
        print(f"  {h['articles']:>8,}  {h['heading']}  ({levels}){extra}")
    print(f"\nMost frequent templates (articles using each; {r['template_redirects']:,} template "
          "redirects merged, so counts with aliases are upper bounds):")
    for t in r["top_templates"]:
        more = len(t["aliases"]) - 5
        extra = f"   [also {', '.join(t['aliases'][:5])}{f' +{more}' if more > 0 else ''}]" if t["aliases"] else ""
        print(f"  {t['articles']:>8,}  {t['template']}{extra}")

## Run

Paths are relative to this notebook. Set `PAGE_PROPS_PATH` once `hiwiki-20261001-page_props.sql.gz` is downloaded. The full report is saved to `JSON_PATH` before it is printed.

In [25]:
DUMP_PATH = "../Data/hiwiki-2026-10-01-p1p1619621.xml.bz2"
PAGE_PROPS_PATH = None     # e.g. "../Data/hiwiki-20261001-page_props.sql.gz"
TOP = 60                   # how many headings/templates to list
DUP_MIN = 10               # smallest group of articles with one skeleton that is reported
JSON_PATH = "survey.json"  # None to skip saving

In [26]:
disambiguation_ids = read_disambiguation_ids(PAGE_PROPS_PATH) if PAGE_PROPS_PATH else set()
report = survey(DUMP_PATH, disambiguation_ids, TOP, DUP_MIN)
if JSON_PATH:
    with open(JSON_PATH, "w", encoding="utf-8") as fh:
        json.dump(report, fh, ensure_ascii=False, indent=1)
print_report(report)

  ... 50,000 pages read (22s)
  ... 100,000 pages read (29s)
  ... 150,000 pages read (36s)
  ... 200,000 pages read (53s)
  ... 250,000 pages read (59s)
  ... 300,000 pages read (66s)
  ... 350,000 pages read (73s)
  ... 400,000 pages read (80s)
  ... 450,000 pages read (86s)
  ... 500,000 pages read (91s)
  ... 550,000 pages read (97s)
  ... 600,000 pages read (105s)
  ... 650,000 pages read (111s)
  ... 700,000 pages read (118s)
  ... 750,000 pages read (123s)
  ... 800,000 pages read (129s)
  ... 850,000 pages read (134s)
  ... 900,000 pages read (139s)
  ... 950,000 pages read (144s)
  ... 1,000,000 pages read (150s)
  ... 1,050,000 pages read (157s)
  ... 1,100,000 pages read (162s)
  ... 1,150,000 pages read (168s)
  ... 1,200,000 pages read (173s)
  ... 1,250,000 pages read (180s)
  ... 1,300,000 pages read (187s)
  ... 1,350,000 pages read (194s)



Dump: ../Data/hiwiki-2026-10-01-p1p1619621.xml.bz2  (207 s)
Wiki hiwiki, schema http://www.mediawiki.org/xml/export-0.11/
Pages in total: 1,393,906  (page IDs 1 to 1,619,621)

Pages per namespace:
    862,034  3 सदस्य वार्ता
    246,795  0 (articles)
    113,193  1 वार्ता
     45,378  2 सदस्य
     37,981  14 श्रेणी
     32,448  15 श्रेणी वार्ता
     26,764  10 साँचा
      8,993  11 साँचा वार्ता
      7,145  4 विकिपीडिया
      4,643  6 चित्र
      2,205  100 प्रवेशद्वार
      1,892  828 मॉड्यूल
      1,011  8 मीडियाविकि
        881  101 प्रवेशद्वार वार्ता
        830  5 विकिपीडिया वार्ता
        801  9 मीडियाविकि वार्ता
        776  7 चित्र वार्ता
         90  12 सहायता
         28  829 मॉड्यूल वार्ता
         18  13 सहायता वार्ता

Namespace 0:
  pages               246,795
  redirects           70,976
  non-redirect        175,819
  empty text          9
  articles            175,810
    redirect wikitext but no <redirect> element: 4
  content models      {'wikitext': 246795}
  revisi